<a href="https://colab.research.google.com/github/obieshka/Python-2025-/blob/hw_5/%D0%BF%D1%80%D0%B0%D0%BA5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import pandas as pd
import bottleneck as bn

# --------------------- Часть 1: Polars ---------------------
print("="*50)
print("Часть 1: Анализ данных с Polars (train.csv)")

df_train = pl.read_csv('train.csv')

print("\nТипы данных столбцов:")
print(df_train.dtypes)

print("\nКоличество пропусков по столбцам:")
print(df_train.null_count())

print("\nСтатистическое описание числовых столбцов:")
print(df_train.describe())

print("\nКоличество пассажиров по классам (Pclass):")
print(df_train['Pclass'].value_counts().sort('Pclass'))

print("\nКоличество выживших по полу:")
survived_by_sex = df_train.filter(pl.col('Survived') == 1) \
                          .group_by('Sex') \
                          .agg(pl.count().alias('count'))
print(survived_by_sex)

print("\nПервые 5 строк таблицы с пассажирами старше 44 лет:")
print(df_train.filter(pl.col('Age') > 44).head())

# --------------------- Часть 2: Pandas + bottleneck ---------------------
print("\n" + "="*50)
print("Часть 2: Pandas + bottleneck (train.csv)")

df_pd = pd.read_csv('train.csv')

mean_age = bn.nanmean(df_pd['Age'].values)
std_age = bn.nanstd(df_pd['Age'].values)
print(f"Средний возраст (bottleneck): {mean_age:.2f}")
print(f"Стандартное отклонение возраста (bottleneck): {std_age:.2f}")


df_pd['Fare_new'] = df_pd['Fare'].apply(lambda x: x * 1.3)
print("Столбец Fare_new создан. Первые 5 значений:")
print(df_pd[['Fare', 'Fare_new']].head())

# --------------------- Часть 3: Оптимизация типов в Housing.csv ---------------------
print("\n" + "="*50)
print("Часть 3: Оптимизация памяти для Housing.csv")


df_housing = pd.read_csv('Housing.csv')
print("Размер памяти ДО оптимизации:")
print(df_housing.info(memory_usage='deep'))
print(f"Общее потребление памяти: {df_housing.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")

# --------------------- Анализ и оптимизация типов ---------------------
print("\nИзменение типов...")


# Перед изменением типов полезно посмотреть на уникальные значения и диапазоны:
# print(df_housing.describe(include='all'))
# print(df_housing.nunique())

# 1. price – цена дома (целые числа, от ~1 750 000 до ~13 300 000)
#    Для int32 хватит (макс. ~2e9), int64 избыточен.
df_housing['price'] = df_housing['price'].astype('int32')

# 2. area – площадь дома (целые, от ~1650 до ~16200) → int32
df_housing['area'] = df_housing['area'].astype('int32')

# 3. bedrooms – число спален (от 1 до 6) → int8 (диапазон -128..127)
df_housing['bedrooms'] = df_housing['bedrooms'].astype('int8')

# 4. bathrooms – число ванных (от 1 до 4) → int8
df_housing['bathrooms'] = df_housing['bathrooms'].astype('int8')

# 5. stories – количество этажей (от 1 до 4) → int8
df_housing['stories'] = df_housing['stories'].astype('int8')

# 6. parking – количество парковочных мест (от 0 до 3) → int8
df_housing['parking'] = df_housing['parking'].astype('int8')

# 7. Бинарные столбцы: mainroad, guestroom, basement, hotwaterheating,
#    airconditioning, prefarea – принимают только 'yes'/'no'.
#    Лучше всего преобразовать в bool (занимает 1 байт, а не объект).
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
               'airconditioning', 'prefarea']
for col in binary_cols:
    df_housing[col] = df_housing[col].map({'yes': True, 'no': False}).astype('bool')

# 8. furnishingstatus – категориальный признак с тремя значениями:
#    'furnished', 'semi-furnished', 'unfurnished'.
#    Используем тип category (экономит память по сравнению с object).
df_housing['furnishingstatus'] = df_housing['furnishingstatus'].astype('category')

print("Размер памяти ПОСЛЕ оптимизации:")
print(df_housing.info(memory_usage='deep'))
print(f"Общее потребление памяти: {df_housing.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")
print("\nЭкономия памяти достигнута за счёт использования более компактных типов (int32/int8 вместо int64, bool и category).")

Часть 1: Анализ данных с Polars (train.csv)

Типы данных столбцов:
[Int64, Int64, Int64, String, String, Float64, Int64, Int64, String, Float64, String, String]

Количество пропусков по столбцам:
shape: (1, 12)
┌─────────────┬──────────┬────────┬──────┬───┬────────┬──────┬───────┬──────────┐
│ PassengerId ┆ Survived ┆ Pclass ┆ Name ┆ … ┆ Ticket ┆ Fare ┆ Cabin ┆ Embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---  ┆   ┆ ---    ┆ ---  ┆ ---   ┆ ---      │
│ u32         ┆ u32      ┆ u32    ┆ u32  ┆   ┆ u32    ┆ u32  ┆ u32   ┆ u32      │
╞═════════════╪══════════╪════════╪══════╪═══╪════════╪══════╪═══════╪══════════╡
│ 0           ┆ 0        ┆ 0      ┆ 0    ┆ … ┆ 0      ┆ 0    ┆ 687   ┆ 2        │
└─────────────┴──────────┴────────┴──────┴───┴────────┴──────┴───────┴──────────┘

Статистическое описание числовых столбцов:
shape: (9, 13)
┌────────────┬─────────────┬──────────┬──────────┬───┬───────────┬───────────┬───────┬──────────┐
│ statistic  ┆ PassengerId ┆ Survived ┆ Pclass   ┆ … ┆ Tic

/tmp/ipykernel_766/3229618436.py:26: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias('count'))
